# A0: Setup & Verbindungstest

**Ziel:** Zugangsdaten eintragen, Verbindung zur Infrastruktur prüfen und die eigene Datenbanktabelle anlegen.
Das Datenbankschema ist vom Dozenten bereits angelegt.

## Schritte
1. `config.py` anpassen (`GROUP_ID`, `GROUP_PASSWORD`, `STATION_ID`, `FUEL_TYPE`)
2. PostgreSQL-Verbindung testen
3. Tankstelle wählen und `STATION_ID` setzen
4. `predictions`-Tabelle anlegen
5. MLflow-Verbindung testen

## 0. Abhängigkeiten installieren

Die nächste Zelle installiert alle benötigten Python-Pakete. Führen Sie diese **einmalig** aus.

In [ ]:
import sys, os

# Pakete mit kurzer Beschreibung
packages = {
    # Daten & ML
    "pandas==2.2.2":           "Tabellarische Datenverarbeitung (DataFrames)",
    "numpy==1.26.4":           "Numerische Berechnungen (Arrays, Matrizen)",
    "scikit-learn==1.5.1":     "Machine-Learning-Algorithmen (RandomForest, Metriken)",

    # Experiment Tracking & Modell-Registry
    "mlflow==2.14.3":          "Experiment-Tracking, Modell-Registry und -Deployment",

    # Datenversionierung
    "dvc==3.51.2":             "Data Version Control – Pipelines und Datensätze versionieren",
    "dvc-s3==3.2.0":           "DVC S3-Plugin – MinIO/S3 als Remote-Speicher (dvc push/pull)",
    "boto3==1.34.106":         "AWS/S3-Client – fixiert für Kompatibilität mit aiobotocore 2.13.0",
    "aiobotocore==2.13.0":     "Async S3-Client (fixiert auf 2.x – ab 3.x kein boto3-Extra)",
    "s3fs==2024.2.0":          "S3-Dateisystem-Interface (kompatibel mit aiobotocore 2.13.0)",
    "pathspec==0.11.2":        "Pfadmuster-Matching (interne DVC-Abhängigkeit, fixierte Version)",

    # Jupyter
    "jupyterlab==4.2.5":       "Web-basierte Entwicklungsumgebung für Notebooks",
    "nbconvert==7.16.4":       "Notebooks als Skripte ausführen (für DVC-Pipelines)",
    "ipykernel==6.29.5":       "Python-Kernel für Jupyter",

    # API
    "fastapi==0.111.0":        "Modernes Web-Framework für die Prediction-API (A4)",
    "uvicorn[standard]==0.30.1": "ASGI-Server zum Starten der FastAPI-App",

    # Monitoring
    "evidently==0.4.30":       "Data-Drift- und Modell-Monitoring-Reports (A5)",

    # Datenbank
    "psycopg2-binary==2.9.9":  "PostgreSQL-Treiber für Python",
    "sqlalchemy==2.0.31":      "SQL-Toolkit (wird intern von MLflow genutzt)",

    # Sonstiges
    "pyyaml==6.0.1":           "YAML-Parser (wird intern von DVC genutzt)",
    "scipy>=1.12":             "Wissenschaftliche Berechnungen (Statistiktests)",

    # Tests
    "pytest==8.2.2":           "Test-Framework für automatisierte Bewertung",
    "httpx==0.27.0":           "HTTP-Client zum Testen der FastAPI-Endpoints",
}

# Im JupyterHub sind alle Pakete bereits im Docker-Image vorinstalliert
if os.environ.get("JUPYTERHUB_USER"):
    print("JupyterHub erkannt – Pakete sind bereits im Image vorinstalliert:")
    for pkg, desc in packages.items():
        print(f"  {pkg:<40}  # {desc}")
    print()
    print("✓ Keine Installation nötig.")
else:
    print("Installiere Pakete ...
")
    for pkg, desc in packages.items():
        name = pkg.split("=")[0].split(">")[0].split("<")[0]
        print(f"  {pkg:<40}  # {desc}")
    print()
    import subprocess
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install"] + list(packages.keys()),
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✓ Alle Pakete erfolgreich installiert.")
    else:
        print("✗ Fehler bei der Installation:")
        print(result.stderr[-2000:])


## 1. Konfiguration prüfen

Öffnen Sie `config.py` im Projektverzeichnis und tragen Sie **vier Werte** ein:

```python
GROUP_ID       = "gruppe_03"      # ← Ihre Gruppe (vom Dozenten)
GROUP_PASSWORD = "AbcXyz9..."    # ← Ihr Passwort (vom Dozenten per E-Mail)
STATION_ID     = "STATION_UUID"  # ← UUID Ihrer Tankstelle (siehe Schritt 3)
FUEL_TYPE      = "e5"            # ← Kraftstoffsorte: "e5", "e10" oder "diesel"
```

> **Wichtig:** `GROUP_ID` wird als PostgreSQL-Schemaname verwendet.  
> Nur Kleinbuchstaben, Ziffern und Unterstriche – exakt wie vom Dozenten angegeben.

> **Nach jeder Änderung an `config.py`:** Kernel neu starten!  
> Menü → **Kernel → Restart Kernel…** – sonst bleiben alte Werte im Speicher.

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from config import GROUP_ID, GROUP_PASSWORD, STATION_ID, FUEL_TYPE, SERVER_IP, \
                   MLFLOW_TRACKING_URI, GASPRICES_SCHEMA, DB_CONFIG, \
                   MINIO_ACCESS_KEY, MINIO_SECRET_KEY, MINIO_ENDPOINT

print(f"GROUP_ID            : {GROUP_ID}")
print(f"GROUP_PASSWORD      : {'✓ gesetzt' if GROUP_PASSWORD != 'ChangeMe' else '✗ noch nicht gesetzt!'}")
print(f"STATION_ID          : {STATION_ID}")
print(f"FUEL_TYPE           : {FUEL_TYPE}")
print(f"DB Host             : {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"MLFLOW_TRACKING_URI : {MLFLOW_TRACKING_URI}")

if GROUP_PASSWORD == 'ChangeMe':
    print("\n⚠  GROUP_PASSWORD noch nicht gesetzt → config.py anpassen!")

### DVC Remote konfigurieren (einmalig)

DVC speichert Datensätze (`prices.csv`, `features.csv`) zentral in **MinIO** – einem S3-kompatiblen
Object Store auf dem Kurs-Server. So können alle Gruppen ihre Daten versioniert und für andere
sichtbar ablegen.

Führen Sie die folgende Zelle **einmalig** aus. Sie schreibt die Zugangsdaten in `.dvc/config.local`
(diese Datei ist gitignored und verlässt Ihren Rechner nicht).

In [ ]:
import subprocess

# DVC-Remote konfigurieren (schreibt in .dvc/config.local – gitignored, bleibt lokal)
# Jede Gruppe hat ihr eigenes Unterverzeichnis: s3://dvc/<GROUP_ID>/
cmds = [
    ["dvc", "remote", "modify", "minio", "url",               f"s3://dvc/{GROUP_ID}",  "--local"],
    ["dvc", "remote", "modify", "minio", "access_key_id",     MINIO_ACCESS_KEY,         "--local"],
    ["dvc", "remote", "modify", "minio", "secret_access_key", MINIO_SECRET_KEY,         "--local"],
]
for cmd in cmds:
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FEHLER: {result.stderr.strip()}")
    else:
        print(f"OK: dvc remote modify minio {cmd[3]} ...")

print(f"\nDVC Remote konfiguriert:")
print(f"  Endpoint : {MINIO_ENDPOINT}")
print(f"  Pfad     : s3://dvc/{GROUP_ID}/")
print(f"  Nutzer   : {MINIO_ACCESS_KEY}")
print(f"\nSie können jetzt 'dvc push' und 'dvc pull' verwenden.")
print(f"Dateien im Browser sichtbar: http://141.47.5.55:9001")


## 2. PostgreSQL-Verbindung testen

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT version()")
    version = cur.fetchone()[0]
    conn.close()
    print("✓ PostgreSQL-Verbindung erfolgreich!")
    print(f"  {version[:70]}")
except Exception as e:
    print(f"✗ Verbindung fehlgeschlagen: {e}")
    print("  → GROUP_ID und GROUP_PASSWORD in config.py prüfen")
    print("  → VPN / Hochschulnetz aktiv?")

## 3. Tankstelle wählen und STATION_ID setzen

**Schritt 1:** Prüfen Sie in der nächsten Zelle, welche Stationen ausreichend Daten haben.

**Schritt 2:** Wählen Sie eine Tankstelle aus der Liste – nehmen Sie eine mit vielen Einträgen und Daten ab `2025-02-15`.

> **Tipp:** Sie können auch eine Tankstelle Ihrer Wahl über den [Tankstellen-Finder von Tankerkönig](https://creativecommons.tankerkoenig.de/TankstellenFinder/index.html) suchen und deren UUID direkt von der Webseite ablesen.

**Schritt 3:** UUID in `config.py` → `STATION_ID` eintragen und die Verifikationszelle ausführen.

> Bleiben Sie für den gesamten Kursverlauf bei **einer** Tankstelle.

In [ ]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    cur.execute(f"""
        SELECT COUNT(*), MIN(date), MAX(date)
        FROM {GASPRICES_SCHEMA}.gas_station_information_history
    """)
    count, min_date, max_date = cur.fetchone()
    print(f"✓ Lesezugriff auf '{GASPRICES_SCHEMA}'")
    print(f"  Preisänderungen gesamt : {count:,}")
    print(f"  Zeitraum               : {min_date} bis {max_date}")
    print()

    cur.execute(f"""
        SELECT stid::TEXT, COUNT(*) AS eintraege,
               MIN(date)::DATE AS erster_eintrag,
               MAX(date)::DATE AS letzter_eintrag
        FROM {GASPRICES_SCHEMA}.gas_station_information_history
        GROUP BY stid ORDER BY eintraege DESC LIMIT 15
    """)
    rows = cur.fetchall()
    conn.close()

    print("Top-15 Stationen nach Datenmenge:")
    print(f"  {'UUID':<38} {'Einträge':>9}  {'von':<12} {'bis'}")
    print(f"  {'-'*38} {'-'*9}  {'-'*12} {'-'*12}")
    for stid, n, first, last in rows:
        print(f"  {stid}  {n:>9,}  {str(first):<12} {last}")

    print()
    print("→ Wählen Sie eine Station mit vielen Einträgen ab 2025-02-15.")
    print("  Tragen Sie die UUID in config.py → STATION_ID ein.")

except Exception as e:
    print(f"✗ Fehler: {e}")

### STATION_ID in config.py eintragen

Tragen Sie jetzt die UUID Ihrer gewählten Tankstelle in `config.py` → `STATION_ID` ein:

```python
STATION_ID = "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"  # ← Ihre UUID hier
```

> **Nach dem Speichern:** Kernel neu starten!  
> Menü → **Kernel → Restart Kernel…** – dann alle Zellen erneut von oben ausführen.

Führen Sie anschließend die nächste Zelle aus, um die Station zu verifizieren.

In [ ]:
# config.py neu laden, damit die eingetragene STATION_ID wirksam wird
import importlib, config
importlib.reload(config)
from config import STATION_ID, FUEL_TYPE, DB_CONFIG, GASPRICES_SCHEMA

START_DATE = "2025-02-15"

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    cur.execute(f"""
        SELECT COUNT(*) AS eintraege,
               MIN(date)::DATE AS erster_eintrag,
               MAX(date)::DATE AS letzter_eintrag
        FROM {GASPRICES_SCHEMA}.gas_station_information_history
        WHERE stid = %s
    """, (STATION_ID,))
    row = cur.fetchone()

    count, first, last = row
    if count == 0:
        print(f"✗ Keine Daten für STATION_ID: {STATION_ID}")
        print("  → UUID prüfen oder andere Station wählen")
    else:
        has_early_data = first is not None and str(first) <= START_DATE
        print(f"✓ Station gefunden: {STATION_ID}")
        print(f"  Einträge        : {count:,}")
        print(f"  Erster Eintrag  : {first}  {'✓' if has_early_data else f'⚠ Daten erst ab {first}, nicht ab {START_DATE}'}")
        print(f"  Letzter Eintrag : {last}")
        if count < 500:
            print(f"  ⚠ Wenige Einträge – besser eine Station mit mehr Daten wählen")
        else:
            print(f"  ✓ Ausreichend Daten für das Projekt")

    # Prüfen ob die gewählte Kraftstoffsorte Preisdaten hat
    cur.execute(f"""
        SELECT COUNT(*) 
        FROM {GASPRICES_SCHEMA}.gas_station_information_history
        WHERE stid = %s AND {FUEL_TYPE} IS NOT NULL AND {FUEL_TYPE} > 0
    """, (STATION_ID,))
    fuel_count = cur.fetchone()[0]
    print(f"  Preise für {FUEL_TYPE.upper():<8}: {fuel_count:,} Einträge mit gültigem Preis")
    if fuel_count == 0:
        print(f"  ⚠ Keine {FUEL_TYPE.upper()}-Preise für diese Station – anderen FUEL_TYPE oder andere Station wählen")

    conn.close()

except Exception as e:
    print(f"✗ Fehler: {e}")

## 4. `predictions`-Tabelle anlegen

Das Schema `gruppe_XX` existiert bereits. Legen Sie nun die Tabelle an, in der Ihre API-Vorhersagen gespeichert werden (wird in A4 befüllt, in A5 ausgewertet).

In [ ]:
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS \"{GROUP_ID}\".predictions (
            id            SERIAL       PRIMARY KEY,
            prediction_ts TIMESTAMPTZ  NOT NULL,
            predicted_price  FLOAT        NOT NULL,
            actual_price     FLOAT,
            model_version VARCHAR(50),
            station_id    VARCHAR(50),
            created_at    TIMESTAMPTZ  DEFAULT NOW()
        )
    """)
    conn.commit()

    # Ergebnis prüfen
    cur.execute(
        "SELECT column_name, data_type FROM information_schema.columns "
        "WHERE table_schema = %s AND table_name = 'predictions' ORDER BY ordinal_position",
        (GROUP_ID,)
    )
    cols = cur.fetchall()
    conn.close()

    print(f"✓ Tabelle '{GROUP_ID}'.predictions angelegt")
    for col, dtype in cols:
        print(f"  {col:<20} {dtype}")

except Exception as e:
    print(f"✗ Fehler beim Anlegen der Tabelle: {e}")

## 5. MLflow-Verbindung testen

In [ ]:
import mlflow
from config import DEFAULT_PARAMS

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    experiments = mlflow.search_experiments()
    exp_name = DEFAULT_PARAMS["mlflow"]["experiment_name"]
    existing = mlflow.get_experiment_by_name(exp_name)

    print(f"✓ MLflow erreichbar: http://141.47.5.55:5000")
    print(f"  Experimente auf dem Server : {len(experiments)}")
    print(f"  Ihr Experiment '{exp_name}': "
          f"{'bereits vorhanden' if existing else 'wird bei A3 automatisch angelegt'}")
except Exception as e:
    print(f"✗ MLflow nicht erreichbar: {e}")
    print("  → VPN / Hochschulnetz aktiv?")

## ✓ Setup abgeschlossen

Wenn alle Checks grün sind, können Sie mit **A1_Datenerfassung.ipynb** beginnen.

| Check | Zelle |
|---|---|
| `GROUP_ID` und `GROUP_PASSWORD` eingetragen | 3 |
| PostgreSQL-Verbindung ✓ | 4 |
| `STATION_ID` gewählt und eingetragen | 6 |
| `predictions`-Tabelle angelegt ✓ | 8 |
| MLflow erreichbar ✓ | 10 |